In [1]:
import numpy as np
from scipy.integrate import fixed_quad
import os
import plotly.graph_objects as go
from scipy.special import jv
import pandas as pd

In [2]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

/tmp/ipykernel_16789/3003301750.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data_atlas = pd.read_csv(


In [3]:
# === Global Configuration and Constants ===
start_sqrt_s = 1  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

sigma_tot_lst = []
sqrt_s_lst = []
error_lst = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0729

model_params = {
    'atlas': {
        'pl':  {'mg': 0.412, 'a1': 1.652, 'a2': 1.479}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}


In [4]:

# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    factor = q2 + 9 * abs(k ** 2 - q2 / 4)

    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func):
    k = sqrt_s * x
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s

    return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian

def amp_calculation(diff_T, s, epsilon):
    alpha_pomeron = 1.0 + epsilon
    regge_factor = (s / s0) ** alpha_pomeron
    
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


In [5]:
amp_born_lst = []

sqrt_s_lst = []

# === Main Function ===
def main():
    global start_sqrt_s
    global sqrt_s

    max_sqrt_s = 13000
    step = 100
    n_points = 10000

    # Using only PL model with ATLAS
    mass_model = 'pl'
    ensemble = 'atlas'

    fig = go.Figure()

    sigma_tot_lst = []
    

    

    m2_func = get_m2_function(mass_model)
    params = model_params[ensemble][mass_model]
    mg, a1, a2 = params['mg'], params['a1'], params['a2']
    epsilon = epsilon_values[ensemble]

    sqrt_s = start_sqrt_s
    while sqrt_s <= max_sqrt_s:
        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func),
                0, 1,
                n=n_points
            )[0]

        integral_value = fixed_quad(
            inner_integral,
            0, 1,
            n=n_points
        )[0]

        diff_T = integral_value
        s = sqrt_s * sqrt_s

        

        amp_value = amp_calculation(diff_T, s, epsilon)
        sigma_tot_value = sigma_tot(amp_value, s)

        sigma_tot_lst.append(sigma_tot_value)
        sqrt_s_lst.append(sqrt_s)
        amp_born_lst.append(amp_value)

        sqrt_s += step

    # Add PL model trace
    fig.add_trace(go.Scatter(
        x=sqrt_s_lst,
        y=sigma_tot_lst,
        mode='lines+markers',
        line=dict(
            color='blue',
            width=2
        ),
        marker=dict(
            size=4
        ),
        name='PL Model (ATLAS)'
    ))

    # Add ATLAS data
    fig.add_trace(go.Scatter(
        x=x_atlas,
        y=y_atlas,
        mode='markers',
        marker=dict(
            color='black',
            size=6,
            symbol='square'
        ),
        error_y=dict(
            type='data',
            array=y_error_atlas,
            visible=True
        ),
        name='ATLAS Data'
    ))

    # Configure layout
    fig.update_layout(
        title='Sigma Tot vs. sqrt(s) - PL Model with ATLAS Data',
        xaxis=dict(
            title='sqrt(s) [GeV]',
            type='log',
        ),
        yaxis=dict(
            title='Sigma Tot [mb]',
        ),
        showlegend=True,
        legend=dict(
            title='Model/Data'
        ),
        plot_bgcolor='white',
        hovermode='x unified'
    )
    
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

    # fig.show(renderer="browser")
    # fig.write_html("results/sigma_tot/sigma_tot_pl_atlas.html")
    # fig.write_image("results/sigma_tot/sigma_tot_pl_atlas.pdf", width=1200, height=600)


In [6]:
if __name__ == "__main__":
    main()

s_lst = []
for key, value in enumerate(sqrt_s_lst):
    s_lst.append(value ** 2)
    # print(f"sqrt(s) = {value:.2f} GeV, s = {s_lst[key]:.2f} GeV^2")

In [10]:
print(amp_born_lst)
print(s_lst)

[63.115184501271365j, 1347190.7176672095j, 5898675.860122345j, 14030218.736312661j, 25964720.860653378j, 41866654.51598633j, 61867890.781103455j, 86079194.04572102j, 114596439.07204361j, 147504383.4448267j, 184879140.67330095j, 226789894.8495602j, 273300141.1816686j, 324468614.13086736j, 380350000.93986464j, 440995502.63094366j, 506453283.4883672j, 576768837.0409378j, 651985288.230332j, 732143645.9378079j, 817283016.2906829j, 907440784.5550759j, 1002652771.5593326j, 1102953369.2413766j, 1208375658.9160106j, 1318951515.1110897j, 1434711697.2539096j, 1555685931.0526915j, 1681902981.0784702j, 1813390715.7857614j, 1950176165.9985693j, 2092285577.7186646j, 2239744459.976163j, 2392577628.331045j, 2550809244.54304j, 2714462852.852071j, 2883561413.249029j, 3058127332.064564j, 3238182490.1598883j, 3423748268.9666686j, 3614845574.591939j, 3811494860.1772485j, 4013716146.6786423j, 4221529042.2143965j, 4434952760.11078j, 4654006135.761345j, 4878707642.402831j, 5109075405.899547j, 5345127218.618629

In [7]:
def eikonal_int(b, q, amp_born):
    return  q * jv(0, q * b) * amp_born

def amp_eik_int(b, q, eik):
    return b * jv(0, q * b) * (1 - np.exp(eik * 1j))

eikonal_int_lst = []
eikonal_lst = []

amp_eik_int_lst = []
amp_eik_lst = []

b_lst = list(np.linspace(0, 30, 130))  
q_lst = list(np.linspace(0, 30, 130))  

# amp_eik_int_value, _ = fixed_quad(
#     lambda b: amp_eik_int(b, 0, eikonal_value), 0, 30, n = 1000
# )
# amp_eik_int_lst.append(amp_eik_int_value)
# amp_eik_value = amp_eik_int_value * 1j * s_value
# amp_eik_lst.append(amp_eik_value.imag)
# # print(amp_eik_value.imag)


In [8]:
def eikonal_int(b, q, amp_born):
    return  q * jv(0, q * b) * amp_born

def amp_eik_int(b, q, eik):
    return b * jv(0, q * b) * (1 - np.exp(eik * 1j))

eikonal_int_lst = []
eikonal_lst = []

amp_eik_int_lst = []
amp_eik_lst = []

b_lst = list(np.linspace(0, 30, 130))  
q_lst = list(np.linspace(0, 30, 130))  

for idx, (q_value, b_value, amp_born_value, s_value) in enumerate(zip(q_lst, b_lst, amp_born_lst, s_lst)):
    if idx < len(q_lst) - 1:  

        next_q = q_lst[idx+1]
        next_b = b_lst[idx+1]

        eikonal_int_value, _= fixed_quad(
            lambda q: eikonal_int(b_value, q, amp_born_value), q_value, next_q, n = 1000
        )
        print(f'lower = {q_value}, upper = {next_q}')
        print(f'amp = {amp_born_value}, s = {s_value}')
        print(f'eikonal integral value = {eikonal_int_value}')
        eikonal_value = eikonal_int_value/s_value
        print(f'eikonal value = {eikonal_value}')
        print('\n')

        amp_eik_int_value, _ = fixed_quad(
            lambda b: amp_eik_int(b, next_q, eikonal_value), 
            b_value, next_b, n = 1000
        )
        print(f'amp eik integral value = {amp_eik_int_value}')
        amp_eik_value = 1j * s_value * amp_eik_int_value
        print(f'amp eik = {amp_eik_value.imag}')
        print(100*'-')
        print('\n')
        

    else:
        pass  
    

lower = 0.0, upper = 0.23255813953488372
amp = 63.115184501271365j, s = 1
eikonal integral value = 1.7067383586065807j
eikonal value = 1.7067383586065807j


amp eik integral value = (0.022126665259198725+0j)
amp eik = 0.022126665259198725
----------------------------------------------------------------------------------------------------


lower = 0.23255813953488372, upper = 0.46511627906976744
amp = 1347190.7176672095j, s = 10201
eikonal integral value = 109091.06083674096j
eikonal value = 10.694153596386721j


amp eik integral value = (0.08053109308692434+0j)
amp eik = 821.4976805797152
----------------------------------------------------------------------------------------------------


lower = 0.46511627906976744, upper = 0.6976744186046512
amp = 5898675.860122345j, s = 40401
eikonal integral value = 782461.5003879297j
eikonal value = 19.367379529910888j


amp eik integral value = (0.12948883231713182+0j)
amp eik = 5231.478314444443
------------------------------------------------

In [ ]:
# def eikonal(b, q, amp_born, s):
#     return  (q * jv(0, q * b) * amp_born) / s

# def amp_eik(b, q, eik, s):
#     return b * jv(0, q * b) * (1 - np.exp(eik * 1j)) * 1j* s

# result_lst = []
# for idx, (b_value, amp_born_value, s_value) in enumerate(zip(b_lst, amp_born_lst, s_lst)):
        
#     result, _ = fixed_quad(
#         lambda b: amp_eik(
#             b,
#             q,  
#             fixed_quad(  
#                 lambda q: eikonal(b, q, amp_born_value, s_value),
#                 0, 30, n=10000
#             )[0],
#             s_value
#         ),
#         0, 30, n=10000
#     )

#     result_lst.append(result)

    
# print(result_lst)

NameError: name 'q' is not defined